# S04 · A calm way to find and fix a bug

Sometimes code runs with no error at all, but the answer is simply wrong. That is
the sneakiest kind of bug. The cure is not cleverness, it is method: change one
thing at a time, then check, exactly like a careful science experiment. We take a
tiny function that gives the wrong answer, find the bug with `print()`, fix it, and
then use `assert` to guard our assumptions.

**New here? Read this once.**

- New to Python? You can do this whole notebook. Press play on each cell, top to
  bottom, and read the note above it. The method matters more than the code.
- The method in one line: **reproduce, read, isolate, fix one thing, verify.**
- Stuck on a word? It is in `primers/glossary.md`.
- Already comfortable? There is a **Stretch (optional)** cell at the end that shows
  the built-in debugger.

## Setup

This notebook uses only built-in Python, so there is nothing to install, on
**Google Colab** or on your **own machine**.

In [ ]:
# Nothing to install for this notebook - it uses only built-in Python.
print("Setup complete - nothing to install.")

## The method in one line

**Reproduce it, read it, isolate the cause, fix one thing, verify it is gone.**
Below we follow exactly that, on a function small enough to hold in your head.

## Step 1 — a tiny function that looks right but is wrong

This function is meant to return the **average** of a list of numbers, say the units
sold each day. Read it and see if you can spot the bug before we run it.

In [ ]:
# This is meant to compute the average (mean) of a list of numbers.
def average(numbers):
    total = 0
    for one_number in numbers:
        total = total + one_number
    # BUG: we divide by 2 every time, instead of by how many numbers there are.
    answer = total / 2
    return answer


# Try it on daily units sold whose average we can work out by hand: (2+4+6)/3 = 4.
units_sold_each_day = [2, 4, 6]
result = average(units_sold_each_day)
print("the function says the average is:", result)
print("but by hand we know the average of 2, 4, 6 is 4.0")

## Step 2 — reproduce the problem reliably

The function returns `6.0`, not `4.0`. Good news, oddly: the bug happens every time,
so we can study it. A bug you can reproduce is a bug you can fix. A bug that appears
only sometimes is far harder.

## Step 3 — isolate the cause with print()

We add `print(...)` lines inside the function to watch the values as it runs. This
is **print-debugging**: the quickest way to see what your code is actually doing,
and everyone, beginner and expert alike, uses it constantly.

In [ ]:
# Same function, but now with print() lines so we can watch it think.
def average_with_prints(numbers):
    total = 0
    for one_number in numbers:
        total = total + one_number
        print("  added", one_number, "-> running total is now", total)

    print("  final total is", total)
    print("  number of items is", len(numbers))

    answer = total / 2          # the suspicious line
    print("  we divided by 2 and got", answer)
    return answer


print("Running on [2, 4, 6]:")
average_with_prints([2, 4, 6])

### What the prints reveal

The running total reaches `12`, which is correct. But then we divide by `2` and get
`6.0`. We should have divided by the **number of items** (`3`), not by `2`. We have
found the one bad line.

## Step 4 — fix one thing

We change exactly one line: divide by `len(numbers)` instead of by `2`. Changing
only one thing means that if it works, we know *why* it works. If we changed five
things at once and it worked, we would have learned nothing.

In [ ]:
# The fixed version. Only ONE line changed: the division.
def average_fixed(numbers):
    total = 0
    for one_number in numbers:
        total = total + one_number
    # Fixed: divide by how many numbers there are.
    answer = total / len(numbers)
    return answer


print("average of [2, 4, 6] is now:", average_fixed([2, 4, 6]))
print("average of [10, 20] is now:", average_fixed([10, 20]))

## Step 5 — verify the fix

We check the answers against ones we can work out by hand. `(2+4+6)/3 = 4.0` and
`(10+20)/2 = 15.0`. Both match, so the bug is really gone, not just hidden.

In [ ]:
# Compare the function's answer to the answer we know is correct.
print("expected 4.0, got ", average_fixed([2, 4, 6]))
print("expected 15.0, got", average_fixed([10, 20]))

## Step 6 — guard your assumptions with assert

`assert` checks that something you *believe* is true really is. If it is true,
nothing happens and the program carries on. If it is false, Python stops right there
with a clear message, so a wrong assumption can never sneak through unnoticed.

In [ ]:
# A simple assert: we believe the average of 2, 4, 6 is 4.0.
# If this is true, you see nothing - the program just continues.
assert average_fixed([2, 4, 6]) == 4.0

print("The assert passed, so our function agrees with the maths. Good.")

## Step 7 — what a failing assert looks like

To see the other side, here is an assert that is *meant* to fail. We catch it with
`try` / `except` so the notebook keeps running, and print the helpful message we
attached to it.

In [ ]:
# We deliberately assert something false, with a message explaining the check.
try:
    assert average_fixed([2, 4, 6]) == 99.0, "average of 2,4,6 should be 4.0, not 99.0"
except AssertionError as the_error:
    print("The assert failed, as expected. Its message was:")
    print("  ", the_error)

## Using assert to protect a function

A common, friendly use: check the *inputs* at the top of a function. Here we refuse
to average an empty list (which would divide by zero) and we say why. This turns a
confusing crash later into a clear message now.

In [ ]:
# A safer average that checks its assumption before doing any maths.
def safe_average(numbers):
    # We assume the list is not empty. Say so, and explain if it is not.
    assert len(numbers) > 0, "cannot take the average of an empty list"
    total = 0
    for one_number in numbers:
        total = total + one_number
    return total / len(numbers)


# This works fine.
print("safe_average([4, 8]) =", safe_average([4, 8]))

# This one trips the guard. We catch it so the notebook keeps running.
try:
    safe_average([])
except AssertionError as the_error:
    print("Guard worked. Message:", the_error)

### Stretch (optional) — the built-in debugger

Skip this if today is your first week of code. `print()` is enough to catch most
bugs, but Python also has a proper debugger that lets you **pause** a program and
look at every variable, one line at a time. You do not need to install anything.

You drop one line where you want to pause:

```python
def average(numbers):
    total = 0
    for one_number in numbers:
        total = total + one_number
    breakpoint()          # Python pauses HERE and hands you a prompt
    answer = total / 2
    return answer
```

When it pauses, you type short commands at the `(Pdb)` prompt:

- `p total` — **p**rint the value of `total` right now
- `n` — run the **n**ext line
- `l` — **l**ist the code around where you are
- `c` — **c**ontinue running to the end

We describe it here rather than run it, because a `breakpoint()` stops and waits
for you to type, which would freeze this notebook. Try it in your own script once
you are comfortable: it is `print()` with superpowers. The full reference is the
[pdb docs](https://docs.python.org/3/library/pdb.html).

## What you just learned

- Debugging is calm and repeatable: **reproduce, read, isolate, fix one thing,
  verify.**
- **print()** lets you watch what your code is really doing, the fastest first move.
- Change **one thing at a time**, then check, so you always know what fixed it.
- **assert** turns a silent wrong assumption into a loud, clear message. Use it to
  guard inputs and to check your own work.

That is the whole secret: the scientific method, applied to your code.